# Modeling
In this section, multiple models are implemented to perform brain tumor segmentation from MRI images. The goal is to compare different approaches ranging from simple baselines to advanced deep learning architectures.

The following models are explored:
- Baseline (threshold-based segmentation)
- Convolutional Neural Network (CNN)
- U-Net (advanced segmentation model)

The dataset is split into training and validation sets to evaluate model generalization.

In [1]:
import os, sys
import torch
from torch.utils.data import DataLoader, random_split


def _find_root(marker="dataset.py"):
    d = os.path.abspath(os.getcwd())
    while d != os.path.dirname(d):
        if os.path.exists(os.path.join(d, marker)):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.getcwd())


BASE_DIR = _find_root()
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

from dataset import BrainTumorDataset

SEED = 42
BATCH_SIZE = 8
NUM_WORKERS = 0 if os.name == "nt" else 4

dataset = BrainTumorDataset(
    image_dir=os.path.join(BASE_DIR, "data", "processed", "images"),
    mask_dir=os.path.join(BASE_DIR, "data", "processed", "masks"),
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 2451
Validation size: 613


In [2]:
import os
import mlflow
import mlflow.pytorch
from pathlib import Path

os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")

mlruns_path = os.path.join(BASE_DIR, "mlruns")
os.makedirs(mlruns_path, exist_ok=True)

mlflow.set_tracking_uri(Path(mlruns_path).as_uri())
mlflow.set_experiment("Brain-Tumor-Segmentation")
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", mlflow.get_experiment_by_name("Brain-Tumor-Segmentation").name)

MLflow tracking URI: file:///root/mlp/mlruns
Experiment: Brain-Tumor-Segmentation


### Baseline Model

A simple threshold-based segmentation approach is used as a baseline. This method classifies pixels based on intensity values, without learning from data.

This serves as a reference to highlight the limitations of non-learning approaches.

In [3]:
# Baseline Evaluation

from metrics import dice_score, iou_score

def evaluate_baseline(loader):
    dice_total = 0
    iou_total = 0
    
    for images, masks in loader:
        
        if images.shape[1] == 3:
            images_gray = images.mean(dim=1, keepdim=True)
        else:
            images_gray = images
        
        preds = (images_gray > 0.5).float()
        
        dice_total += dice_score(preds, masks)
        iou_total += iou_score(preds, masks)
    
    return dice_total / len(loader), iou_total / len(loader)

baseline_dice, baseline_iou = evaluate_baseline(val_loader)

In [4]:
with mlflow.start_run(run_name="Baseline"):
    mlflow.log_params({
        "model": "Baseline",
        "method": "threshold",
        "threshold": 0.5,
    })
    mlflow.log_metrics({
        "val_dice": float(baseline_dice),
        "val_iou": float(baseline_iou),
    })
    print(f"[Baseline] Dice: {float(baseline_dice):.4f} | IoU: {float(baseline_iou):.4f}")
    print("Baseline run logged to MLflow.")


[Baseline] Dice: 0.0684 | IoU: 0.0358
Baseline run logged to MLflow.


###  Convolutional Neural Network (CNN)

A simple CNN is implemented to learn spatial features from MRI images. While it improves over the baseline, it lacks the ability to precisely localize tumor regions.

In [5]:
import torch
import torch.nn.functional as F
from metrics import dice_score, iou_score, dice_loss
from model import SimpleCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

EPOCHS = 60
PATIENCE = 15
use_amp = False


def evaluate_model(model, loader, device):
    model.eval()
    dice_total, iou_total = 0.0, 0.0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            dice_total += dice_score(preds, masks).item()
            iou_total += iou_score(preds, masks).item()
    return dice_total / len(loader), iou_total / len(loader)


cnn_model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

cnn_ckpt = os.path.join(BASE_DIR, "models", "cnn.pt")
os.makedirs(os.path.dirname(cnn_ckpt), exist_ok=True)

best_dice, no_improve, cnn_epochs_run = -1.0, 0, 0
with mlflow.start_run(run_name="CNN"):
    mlflow.log_params({"model": "CNN", "epochs": EPOCHS, "batch_size": BATCH_SIZE,
                       "optimizer": "Adam", "learning_rate": 1e-3, "loss": "Dice", "amp": use_amp})
    for epoch in range(EPOCHS):
        cnn_model.train()
        total_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                preds = cnn_model(images)
            loss = dice_loss(preds.float(), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        val_dice, val_iou = evaluate_model(cnn_model, val_loader, device)
        scheduler.step(val_dice)
        cnn_epochs_run = epoch + 1
        mlflow.log_metrics({"train_loss": train_loss, "val_dice": val_dice,
                            "val_iou": val_iou, "lr": optimizer.param_groups[0]["lr"]}, step=epoch)
        print(f"[CNN] Epoch {epoch+1}/{EPOCHS} | loss {train_loss:.4f} | val_dice {val_dice:.4f} | val_iou {val_iou:.4f}")

        if val_dice > best_dice:
            best_dice, no_improve = val_dice, 0
            torch.save(cnn_model.state_dict(), cnn_ckpt)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"[CNN] Early stopping at epoch {epoch+1} | best val_dice {best_dice:.4f}")
                break

    cnn_model.load_state_dict(torch.load(cnn_ckpt, map_location=device))
    cnn_dice, cnn_iou = evaluate_model(cnn_model, val_loader, device)
    mlflow.log_metrics({"best_val_dice": cnn_dice, "best_val_iou": cnn_iou})
    mlflow.pytorch.log_model(cnn_model, name="cnn_model")

print(f"[CNN] Best Val Dice: {cnn_dice:.4f} | Val IoU: {cnn_iou:.4f} | epochs run: {cnn_epochs_run}")

Using device: cuda


[CNN] Epoch 1/60 | loss 0.9146 | val_dice 0.1094 | val_iou 0.0583


[CNN] Epoch 2/60 | loss 0.8755 | val_dice 0.1417 | val_iou 0.0776


[CNN] Epoch 3/60 | loss 0.8611 | val_dice 0.1675 | val_iou 0.0950


[CNN] Epoch 4/60 | loss 0.8492 | val_dice 0.1655 | val_iou 0.0935


...

[CNN] Epoch 40/60 | loss 0.8315 | val_dice 0.1708 | val_iou 0.0964
[CNN] Early stopping at epoch 40 | best val_dice 0.1732


2026/06/14 17:43:07 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/06/14 17:43:07 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


2026/06/14 17:43:10 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


[CNN] Best Val Dice: 0.1732 | Val IoU: 0.0981 | epochs run: 40


###  U-Net (Advanced Model)

U-Net is a specialized architecture for medical image segmentation. Its encoder-decoder structure with skip connections enables precise localization of tumor regions.

This model is expected to outperform both the baseline and CNN models.

In [6]:
from model import UNet

unet_model = UNet().to(device)
optimizer = torch.optim.Adam(unet_model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

unet_ckpt = os.path.join(BASE_DIR, "models", "unet.pt")
os.makedirs(os.path.dirname(unet_ckpt), exist_ok=True)

best_dice, no_improve, unet_epochs_run = -1.0, 0, 0
with mlflow.start_run(run_name="U-Net"):
    mlflow.log_params({"model": "U-Net", "epochs": EPOCHS, "batch_size": BATCH_SIZE,
                       "optimizer": "Adam", "learning_rate": 1e-4, "loss": "Dice", "amp": use_amp})
    for epoch in range(EPOCHS):
        unet_model.train()
        total_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                preds = unet_model(images)
            loss = dice_loss(preds.float(), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        val_dice, val_iou = evaluate_model(unet_model, val_loader, device)
        scheduler.step(val_dice)
        unet_epochs_run = epoch + 1
        mlflow.log_metrics({"train_loss": train_loss, "val_dice": val_dice,
                            "val_iou": val_iou, "lr": optimizer.param_groups[0]["lr"]}, step=epoch)
        print(f"[U-Net] Epoch {epoch+1}/{EPOCHS} | loss {train_loss:.4f} | val_dice {val_dice:.4f} | val_iou {val_iou:.4f}")

        if val_dice > best_dice:
            best_dice, no_improve = val_dice, 0
            torch.save(unet_model.state_dict(), unet_ckpt)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"[U-Net] Early stopping at epoch {epoch+1} | best val_dice {best_dice:.4f}")
                break

    unet_model.load_state_dict(torch.load(unet_ckpt, map_location=device))
    unet_dice, unet_iou = evaluate_model(unet_model, val_loader, device)
    mlflow.log_metrics({"best_val_dice": unet_dice, "best_val_iou": unet_iou})
    mlflow.pytorch.log_model(unet_model, name="unet_model")

print(f"[U-Net] Best Val Dice: {unet_dice:.4f} | Val IoU: {unet_iou:.4f} | epochs run: {unet_epochs_run}")
print("Best weights saved to", unet_ckpt)

[U-Net] Epoch 1/60 | loss 0.8948 | val_dice 0.4804 | val_iou 0.3220


[U-Net] Epoch 2/60 | loss 0.7887 | val_dice 0.5179 | val_iou 0.3562


[U-Net] Epoch 3/60 | loss 0.6195 | val_dice 0.5891 | val_iou 0.4271


[U-Net] Epoch 4/60 | loss 0.4584 | val_dice 0.6838 | val_iou 0.5284


[U-Net] Epoch 5/60 | loss 0.3742 | val_dice 0.6844 | val_iou 0.5321


...

[U-Net] Epoch 58/60 | loss 0.0644 | val_dice 0.8379 | val_iou 0.7277
[U-Net] Early stopping at epoch 58 | best val_dice 0.8415


2026/06/14 17:56:03 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


2026/06/14 17:56:03 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


2026/06/14 17:56:06 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu130) contains a local version label (+cu130). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


[U-Net] Best Val Dice: 0.8415 | Val IoU: 0.7330 | epochs run: 58
Best weights saved to /root/mlp/models/unet.pt


##  Evaluation

Models are evaluated using Dice Score and Intersection over Union (IoU), which are suitable metrics for segmentation tasks with class imbalance.

In [7]:
# Comparison Table

import pandas as pd

results = {
    "Model": ["Baseline", "CNN", "U-Net"],
    "Dice Score": [
        baseline_dice.item(),
        cnn_dice,
        unet_dice
    ],
    "IoU Score": [
        baseline_iou.item(),
        cnn_iou,
        unet_iou
    ]
}

df = pd.DataFrame(results)
df

,Model,Dice Score,IoU Score
0,Baseline,0.068376,0.035834
1,CNN,0.173213,0.098061
2,U-Net,0.841526,0.733048


In [8]:
from datetime import datetime

results_path = os.path.join(BASE_DIR, "RESULTS.txt")
with open(results_path, "w", encoding="utf-8") as f:
    f.write("BRAIN TUMOR SEGMENTATION - RUN RESULTS\n")
    f.write("=" * 45 + "\n")
    f.write(f"Generated : {datetime.now():%Y-%m-%d %H:%M}\n")
    f.write(f"Device    : {device} | AMP: {use_amp}\n")
    f.write(f"Dataset   : {len(dataset)} samples ({len(train_dataset)} train / {len(val_dataset)} val)\n")
    f.write(f"Config    : batch_size={BATCH_SIZE} | epoch_cap={EPOCHS} | seed={SEED}\n\n")
    f.write("Epochs actually run (early stopping):\n")
    f.write(f"  CNN   : {cnn_epochs_run}\n")
    f.write(f"  U-Net : {unet_epochs_run}\n\n")
    f.write("Validation results (best checkpoint):\n")
    f.write(df.to_string(index=False) + "\n\n")
    f.write(f"Best model : U-Net | Dice {unet_dice:.4f} | IoU {unet_iou:.4f}\n")
    f.write(f"Weights    : {unet_ckpt}\n")
    f.write("\nPer-epoch metrics and all runs are tracked in MLflow (mlruns/).\n")

print("Results written to", results_path)
print(open(results_path, encoding="utf-8").read())

Results written to /root/mlp/RESULTS.txt
BRAIN TUMOR SEGMENTATION - RUN RESULTS
Generated : 2026-06-14 17:56
Device    : cuda | AMP: False
Dataset   : 3064 samples (2451 train / 613 val)
Config    : batch_size=8 | epoch_cap=60 | seed=42

Epochs actually run (early stopping):
  CNN   : 40
  U-Net : 58

Validation results (best checkpoint):
   Model  Dice Score  IoU Score
Baseline    0.068376   0.035834
     CNN    0.173213   0.098061
   U-Net    0.841526   0.733048

Best model : U-Net | Dice 0.8415 | IoU 0.7330
Weights    : /root/mlp/models/unet.pt

Per-epoch metrics and all runs are tracked in MLflow (mlruns/).



###  Results Interpretation

The baseline method achieved very low performance, as it relies solely on pixel intensity and cannot effectively distinguish tumor regions from background.

The CNN model showed moderate improvement, demonstrating the ability to learn spatial features. However, its performance remains limited due to its relatively simple architecture and lack of precise localization capability.

The U-Net model significantly outperformed both the baseline and CNN models, achieving a Dice Score of approximately 0.70. This improvement is attributed to its encoder-decoder architecture with skip connections, which enables effective capture of both global context and fine-grained spatial details.

These results highlight the importance of specialized architectures like U-Net for medical image segmentation tasks, particularly in handling class imbalance and accurately localizing tumor regions.

In [9]:
print(masks.min(), masks.max())

tensor(0., device='cuda:0') tensor(1., device='cuda:0')


In [10]:
images, masks = next(iter(train_loader))
print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

Images shape: torch.Size([8, 3, 256, 256])
Masks shape: torch.Size([8, 1, 256, 256])


In [11]:
print("Mask min:", masks.min().item())
print("Mask max:", masks.max().item())

Mask min: 0.0
Mask max: 1.0


In [12]:
images, masks = next(iter(val_loader))
images = images.to(device)

unet_model.eval()
with torch.no_grad():
    preds = unet_model(images)

print("Pred min:", preds.min().item())
print("Pred max:", preds.max().item())

Pred min: 2.810313830606259e-12
Pred max: 1.0
